## Renaming Spark Data Frame columns or expressions using alias

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Understanding lit function in Spark").getOrCreate()

import datetime
users_list = [
    {
        'id': 1,
        'first_name': 'John',
        'last_name': 'Doe',
        'email': 'john.doe@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 54 7400113', 'PAK_Mobile': '+92 300 5638080'},
        'is_customer': True,
        'amount_paid': 123.45,
        'customer_from': datetime.date(2023, 1, 15),  # Date in YYYY-MM-DD format
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)  # Timestamp in YYYY-MM-DD HH:MM:SS format
    },
    {
        'id': 2,
        'first_name': 'Jane',
        'last_name': 'Smith',
        'email': 'jane.smith@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666666', 'PAK_Mobile': '+92 300 0000000'},
        'is_customer': True,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    },
    {
        'id': 3,
        'first_name': 'Jack',
        'last_name': 'Alpha',
        'email': 'jack.alpha@example.com',
        'phone_numer' : {'UAE_Mobile': '+971 99 7666555'},
        'is_customer': False,
        'amount_paid': 0.0,
        'customer_from': datetime.date(2022, 11, 10),
        'last_updated_ts': datetime.datetime(2024, 1, 21, 20, 5, 46)
    }
    # Add more user dictionaries as needed
]

24/03/03 13:16:45 WARN Utils: Your hostname, Qasim resolves to a loopback address: 127.0.1.1; using 172.30.54.131 instead (on interface eth0)
24/03/03 13:16:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/03/03 13:16:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/03/03 13:16:48 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/03/03 13:16:48 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/03/03 13:16:48 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/03/03 13:16:48 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [2]:
from pyspark.sql.functions import concat, lit, col

In [4]:
from pyspark import Row

In [8]:
users_df = spark.createDataFrame([Row(**users) for users in users_list])
users_df

DataFrame[id: bigint, first_name: string, last_name: string, email: string, phone_numer: map<string,string>, is_customer: boolean, amount_paid: double, customer_from: date, last_updated_ts: timestamp]

In [10]:
users_df.show()

+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+
| id|first_name|last_name|               email|         phone_numer|is_customer|amount_paid|customer_from|    last_updated_ts|
+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+
|  1|      John|      Doe|john.doe@example.com|{PAK_Mobile -> +9...|       true|     123.45|   2023-01-15|2024-01-21 20:05:46|
|  2|      Jane|    Smith|jane.smith@exampl...|{PAK_Mobile -> +9...|       true|        0.0|   2022-11-10|2024-01-21 20:05:46|
|  3|      Jack|    Alpha|jack.alpha@exampl...|{UAE_Mobile -> +9...|      false|        0.0|   2022-11-10|2024-01-21 20:05:46|
+---+----------+---------+--------------------+--------------------+-----------+-----------+-------------+-------------------+



In [14]:
users_df. \
    select(
        'first_name',
        'last_name'
    ). \
    printSchema()

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)



In [13]:
users_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_numer: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- is_customer: boolean (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- customer_from: date (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)



In [15]:
users_df. \
    select(
        col('first_name'),
        col('last_name')
    ). \
    printSchema()

root
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)



In [23]:
users_df. \
    select(
        col('id'),
        col('first_name'),
        col('last_name')
    ). \
    withColumn('user_full_name', concat(col('first_name'), lit(', '), col('last_name'))). \
    show(truncate=False)

+---+----------+---------+--------------+
|id |first_name|last_name|user_full_name|
+---+----------+---------+--------------+
|1  |John      |Doe      |John, Doe     |
|2  |Jane      |Smith    |Jane, Smith   |
|3  |Jack      |Alpha    |Jack, Alpha   |
+---+----------+---------+--------------+



In [24]:
users_df. \
    select(
        col('id'),
        col('first_name'),
        col('last_name')
    ). \
    withColumn('user_full_name', concat(users_df['first_name'], lit(', '), users_df['last_name'])). \
    show(truncate=False)

+---+----------+---------+--------------+
|id |first_name|last_name|user_full_name|
+---+----------+---------+--------------+
|1  |John      |Doe      |John, Doe     |
|2  |Jane      |Smith    |Jane, Smith   |
|3  |Jack      |Alpha    |Jack, Alpha   |
+---+----------+---------+--------------+



In [25]:
users_df. \
    select(
        col('id'),
        col('first_name').alias('user_first_name'),
        col('last_name').alias('user_last_name')
    ). \
    withColumn('user_full_name', concat(users_df['first_name'], lit(', '), users_df['last_name'])). \
    show(truncate=False)

AnalysisException: [MISSING_ATTRIBUTES.RESOLVED_ATTRIBUTE_MISSING_FROM_INPUT] Resolved attribute(s) "first_name", "last_name" missing from "id", "user_first_name", "user_last_name" in operator !Project [id#0L, user_first_name#128, user_last_name#129, concat(first_name#1, , , last_name#2) AS user_full_name#133]. ;
!Project [id#0L, user_first_name#128, user_last_name#129, concat(first_name#1, , , last_name#2) AS user_full_name#133]
+- Project [id#0L, first_name#1 AS user_first_name#128, last_name#2 AS user_last_name#129]
   +- LogicalRDD [id#0L, first_name#1, last_name#2, email#3, phone_numer#4, is_customer#5, amount_paid#6, customer_from#7, last_updated_ts#8], false


In [27]:
users_df. \
    select(
        col('id'),
        col('first_name').alias('user_first_name'),
        col('last_name').alias('user_last_name')
    ). \
    withColumn('user_full_name', concat(users_df['user_first_name'], lit(', '), users_df['user_last_name'])). \
    show(truncate=False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `user_first_name` cannot be resolved. Did you mean one of the following? [`id`, `first_name`, `last_name`, `email`, `phone_numer`, `is_customer`, `amount_paid`, `customer_from`, `last_updated_ts`].

In [28]:
users_df. \
    select(
        col('id'),
        col('first_name').alias('user_first_name'),
        col('last_name').alias('user_last_name')
    ). \
    withColumn('user_full_name', concat(col('user_first_name'), lit(', '), col('user_last_name'))). \
    show(truncate=False)

+---+---------------+--------------+--------------+
|id |user_first_name|user_last_name|user_full_name|
+---+---------------+--------------+--------------+
|1  |John           |Doe           |John, Doe     |
|2  |Jane           |Smith         |Jane, Smith   |
|3  |Jack           |Alpha         |Jack, Alpha   |
+---+---------------+--------------+--------------+



In [29]:
users_df. \
    select(
        col('id'),
        col('first_name').alias('user_first_name'),
        col('last_name').alias('user_last_name')
    ). \
    withColumn('user_full_name', concat(col('first_name'), lit(', '), col('last_name'))). \
    show(truncate=False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `first_name` cannot be resolved. Did you mean one of the following? [`user_first_name`, `user_last_name`, `id`].;
'Project [id#0L, user_first_name#171, user_last_name#172, concat('first_name, , , 'last_name) AS user_full_name#176]
+- Project [id#0L, first_name#1 AS user_first_name#171, last_name#2 AS user_last_name#172]
   +- LogicalRDD [id#0L, first_name#1, last_name#2, email#3, phone_numer#4, is_customer#5, amount_paid#6, customer_from#7, last_updated_ts#8], false


In [30]:
users_df. \
    withColumn('user_full_name', concat(col('first_name'), lit(', '), col('last_name'))). \
    select(
        col('id'),
        col('first_name').alias('user_first_name'),
        col('last_name').alias('user_last_name'),
        'user_full_name'
    ). \
    show(truncate=False)

+---+---------------+--------------+--------------+
|id |user_first_name|user_last_name|user_full_name|
+---+---------------+--------------+--------------+
|1  |John           |Doe           |John, Doe     |
|2  |Jane           |Smith         |Jane, Smith   |
|3  |Jack           |Alpha         |Jack, Alpha   |
+---+---------------+--------------+--------------+

